In [0]:
import requests
import pandas as pd
from datetime import datetime
from pyspark.sql import SparkSession
import os

spark = SparkSession.builder.getOrCreate()

# Clean Data Lake Directory Structure
RAW_PATH = "./worldbank_data/raw"
PROCESSED_PATH = "./worldbank_data/processed"

os.makedirs(RAW_PATH, exist_ok=True)
os.makedirs(PROCESSED_PATH, exist_ok=True)

print("✓ Data pipeline folder structure initialized inside worldbank_data/")

In [0]:
# Mix of investment-grade, emerging market, and distressed sovereigns
COUNTRIES = [
    ("USA", "United States"),
    ("GBR", "United Kingdom"),
    ("DEU", "Germany"),
    ("JPN", "Japan"),
    ("CHN", "China"),
    ("BRA", "Brazil"),
    ("MEX", "Mexico"),
    ("IND", "India"),
    ("ZAF", "South Africa"),
    ("TUR", "Turkey"),
    ("ARG", "Argentina"),   # serial defaulter
    ("GRC", "Greece"),      # sovereign debt crisis study
]

# World Bank indicator codes
INDICATORS = [
    ("NY.GDP.MKTP.CD",     "gdp_usd",              "GDP (current US$)"),
    ("NY.GDP.MKTP.KD.ZG",  "gdp_growth_pct",       "GDP Growth Rate (annual %)"),
    ("GC.DOD.TOTL.GD.ZS",  "debt_to_gdp_pct",      "Central Government Debt (% of GDP)"),
    ("FP.CPI.TOTL.ZG",     "inflation_pct",        "Inflation, Consumer Prices (annual %)"),
    ("SL.UEM.TOTL.ZS",     "unemployment_pct",     "Unemployment Rate (%)"),
    ("BN.CAB.XOKA.GD.ZS",  "current_account_gdp",  "Current Account Balance (% of GDP)"),
    ("DT.INT.DECT.EX.ZS",  "debt_service_exports", "Debt Service (% of exports)"),
    ("NY.GDP.PCAP.CD",     "gdp_per_capita_usd",   "GDP per Capita (current US$)"),
]

START_YEAR = 2000
END_YEAR = datetime.today().year

In [0]:
def fetch_indicator(indicator_code, country_codes, start, end):
    """Fetch one World Bank indicator for all countries in one API call."""
    codes = ";".join(country_codes)
    url = (
        f"https://api.worldbank.org/v2/country/{codes}"
        f"/indicator/{indicator_code}"
        f"?format=json&date={start}:{end}&per_page=1000"
    )
    r = requests.get(url, timeout=20)
    r.raise_for_status()
    payload = r.json()
    if len(payload) < 2 or not payload[1]:
        return pd.DataFrame()
    rows = []
    for rec in payload[1]:
        rows.append({
            "country_code": rec["country"]["id"],
            "country_name": rec["country"]["value"],
            "year":         int(rec["date"]),
            "value":        rec["value"],
        })
    df = pd.DataFrame(rows)
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    return df.dropna(subset=["value"])

In [0]:
country_codes = [c[0] for c in COUNTRIES]
frames = []

for code, col_name, description in INDICATORS:
    print(f"  Fetching: {description}")
    try:
        df = fetch_indicator(code, country_codes, START_YEAR, END_YEAR)
        if df.empty:
            print("    ⚠  No data returned")
            continue
        df = df.rename(columns={"value": col_name})
        frames.append(df[["country_code", "country_name", "year", col_name]])
        print(f"    ✓  {len(df)} rows")
    except Exception as e:
        print(f"    ✗  ERROR: {e}")

# Merge all indicators into a wide panel
panel = frames[0]
for df in frames[1:]:
    panel = pd.merge(
        panel,
        df[["country_code", "year", df.columns[-1]]],
        on=["country_code", "year"],
        how="outer"
    )
panel = panel.sort_values(["country_code", "year"]).reset_index(drop=True)
print(f"\n✓ Panel shape: {panel.shape[0]} rows × {panel.shape[1]} columns")

In [0]:
today = datetime.today().strftime("%Y%m%d")
raw_csv_path = f"{RAW_PATH}/sovereign_panel_raw_{today}.csv"

# 1. Save the raw, untouched API data to the raw folder
panel.to_csv(raw_csv_path, index=False)
print(f"✓ Raw API snapshot saved to: {raw_csv_path}")

In [0]:
# Create a copy for cleaning
cleaned_panel = panel.copy()

# 1. Forward-fill missing indicators within each country context over time
cleaned_panel = cleaned_panel.sort_values(["country_code", "year"])
cleaned_panel = cleaned_panel.groupby("country_code", group_keys=False).apply(lambda x: x.ffill())

# 2. Add an explicit data quality tracking flag for missing parameters
numeric_cols = ["gdp_growth_pct", "debt_to_gdp_pct", "inflation_pct", "unemployment_pct"]
cleaned_panel["is_complete_record"] = cleaned_panel[numeric_cols].notna().all(axis=1).astype(int)

print("✓ Data Cleaning Complete: Contextual forward-fill applied & data quality flags generated.")

In [0]:
processed_csv_path = f"{PROCESSED_PATH}/sovereign_panel_clean_{today}.csv"

# 1. Save the cleaned data into the processed folder
cleaned_panel.to_csv(processed_csv_path, index=False)
print(f"✓ Cleaned dataset saved to: {processed_csv_path}")

# 2. Add lineage metadata and convert to Spark DataFrame
cleaned_panel["source"] = "WORLD_BANK"
spark_df = spark.createDataFrame(cleaned_panel)

# 3. Save to Delta Table safely with local session fallback
try:
    spark_df.write.mode("overwrite").saveAsTable("bronze.worldbank_sovereign_raw")
    print("✓ Bronze Delta table created successfully!")
except Exception as e:
    print("⚠ DBFS Root restricted. Registering local temporary Spark view instead...")
    spark_df.createOrReplaceTempView("worldbank_sovereign_raw")
    print("✓ Temporary View created: worldbank_sovereign_raw")

print(f"  Final Table Rows: {spark_df.count()} | Unique Countries: {cleaned_panel['country_code'].nunique()}")

In [0]:
latest = (
    panel.sort_values("year")
         .groupby("country_code")
         .last()
         .reset_index()
)

display(
    latest[[
        "country_name", "year", 
        "gdp_growth_pct", "debt_to_gdp_pct", 
        "inflation_pct", "misery_index", 
        "debt_sustainability_signal"
    ]].sort_values("misery_index", ascending=False)
)